# Walk Forward Optimization (WFO)

This notebook performs a rolling walk-forward optimization to evaluate strategy robustness across different market regimes.
It uses the **Fixed Grid Slicing** approach for maximum efficiency.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import vectorbt as vbt
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if project_root not in sys.path:
    sys.path.append(project_root)

from ggTrader.data.kraken.historical_data import KrakenHistoricalData
from ggTrader.core.fast_backtest import FastBacktest
from ggTrader.utils.utils import make_end_anchored_tscv, plot_cv_indices

In [ ]:
# --- Configuration ---
CONSTANTS = {
    "SYMBOLS": None,  # Set to a list like ["BTC/USD", "ETH/USD"] to override JSON
    "SYMBOLS_FILE": "data/top_50_consistent_movers.json",
    "INTERVAL": "4h",
    "START_DATE": "2023-01-01",
    "END_DATE": "2025-06-01",
    "N_SPLITS": 3,
    "TEST_RATIO": 0.334,
    "START_CASH": 10000,
}

print("WFO Configuration loaded.")

In [ ]:
print("Loading data...")
k = KrakenHistoricalData()
data_df = k.get_ohlcv_df(
    CONSTANTS["SYMBOLS"], 
    interval=CONSTANTS["INTERVAL"], 
    start=pd.to_datetime(CONSTANTS["START_DATE"]).tz_localize('UTC'), 
    end=pd.to_datetime(CONSTANTS["END_DATE"]).tz_localize('UTC')
)
print(f"Loaded {len(data_df)} rows for {len(CONSTANTS['SYMBOLS'])} symbols.")

In [ ]:
# Set up TS splits
tscv, test_size, max_train_size = make_end_anchored_tscv(
    n_samples=len(data_df), 
    n_splits=CONSTANTS["N_SPLITS"], 
    test_ratio=CONSTANTS["TEST_RATIO"]
)

print(f"Average Train Size: {max_train_size}, Test Size: {test_size}")

fig, ax = plt.subplots(figsize=(12, 4))
plot_cv_indices(tscv, data_df.index, ax, CONSTANTS["N_SPLITS"])
plt.show()

In [ ]:
# --- Run Vectorized Backtest on ENTIRE Period ---
param_grid = {
    "adx_threshold": list(range(15, 35, 5)),
    "atr_multiplier": list(np.arange(2.0, 4.5, 0.5)),
    "use_dmp_cross": [True, False],
}

print("Running Full Grid Backtest...")
engine = FastBacktest(data_df, param_grid, init_cash=CONSTANTS["START_CASH"])
pf_full = engine.run()
print("Grid search complete. Now iterating splits...")

In [ ]:
fold_results = []
param_names = list(param_grid.keys())

for i, (tr_idx, tt_idx) in enumerate(tscv.split(data_df.index), 1):
    # A. IN-SAMPLE: Find Best Params
    pf_train = pf_full.iloc[tr_idx]
    sharpe_is = pf_train.sharpe_ratio(group_by=param_names)
    best_params = sharpe_is.idxmax()
    
    # B. OUT-OF-SAMPLE: Test Best Params
    pf_test = pf_full.iloc[tt_idx]
    best_pf_test = pf_test.select(best_params, group_by=param_names)
    
    step_sharpe = best_pf_test.sharpe_ratio().mean()
    
    fold_results.append({
        "Fold": i,
        "Train Start": data_df.index[tr_idx[0]],
        "Test Start": data_df.index[tt_idx[0]],
        "Best Params": best_params,
        "IS Sharpe": sharpe_is.max(),
        "OOS Sharpe": step_sharpe,
        "Profit": best_pf_test.total_profit().sum()
    })

wfo_df = pd.DataFrame(fold_results)
print("\nWFO Summary:")
print(tabulate(wfo_df, headers='keys', tablefmt='psql'))